<a href="https://colab.research.google.com/github/alanoudff/financial-sales-analytics/blob/main/sales_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np


file_path = "sales_data.csv"
df = pd.read_csv(file_path)

print("Shape:", df.shape)
display(df.head(3))

Shape: (1000, 14)


,Product_ID,Sale_Date,Sales_Rep,Region,Sales_Amount,Quantity_Sold,Product_Category,Unit_Cost,Unit_Price,Customer_Type,Discount,Payment_Method,Sales_Channel,Region_and_Sales_Rep
0,1052,2023-02-03,Bob,North,5053.97,18,Furniture,152.75,267.22,Returning,0.09,Cash,Online,North-Bob
1,1093,2023-04-21,Bob,West,4384.02,17,Furniture,3816.39,4209.44,Returning,0.11,Cash,Retail,West-Bob
2,1015,2023-09-21,David,South,4631.23,30,Food,261.56,371.40,Returning,0.20,Bank Transfer,Retail,South-David


# **Basic** **overview**


In [5]:

print("\n--- Dtypes ---")
print(df.dtypes)

print("\n--- Summary (numeric) ---")
display(df.describe(include=[np.number]).T)

print("\n--- Summary (categorical) ---")
display(df.describe(include=["object"]).T)


--- Dtypes ---
Product_ID                int64
Sale_Date                object
Sales_Rep                object
Region                   object
Sales_Amount            float64
Quantity_Sold             int64
Product_Category         object
Unit_Cost               float64
Unit_Price              float64
Customer_Type            object
Discount                float64
Payment_Method           object
Sales_Channel            object
Region_and_Sales_Rep     object
dtype: object

--- Summary (numeric) ---


,count,mean,std,min,25%,50%,75%,max
Product_ID,1000.0,1050.12800,29.573505,1001.00,1024.0000,1051.000,1075.000,1100.00
Sales_Amount,1000.0,5019.26523,2846.790126,100.12,2550.2975,5019.300,7507.445,9989.04
Quantity_Sold,1000.0,25.35500,14.159006,1.00,13.0000,25.000,38.000,49.00
Unit_Cost,1000.0,2475.30455,1417.872546,60.28,1238.3800,2467.235,3702.865,4995.30
Unit_Price,1000.0,2728.44012,1419.399839,167.12,1509.0850,2696.400,3957.970,5442.15
Discount,1000.0,0.15239,0.087200,0.00,0.0800,0.150,0.230,0.30



--- Summary (categorical) ---


,count,unique,top,freq
Sale_Date,1000,340,2023-11-14,8
Sales_Rep,1000,5,David,222
Region,1000,4,North,267
Product_Category,1000,4,Clothing,268
Customer_Type,1000,2,New,504
Payment_Method,1000,3,Credit Card,345
Sales_Channel,1000,2,Retail,512
Region_and_Sales_Rep,1000,20,North-Eve,64


#  **Missing** **values** & **Duplicates**

In [16]:
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
missing_report = missing_report[missing_report["missing_count"]> 0 ]
print("\n--- Missing values & Duplicates report ---")
display(missing_report if len(missing_report) else "No missing values")

dup_count = df.duplicated().sum()
print("\nDuplicate rows:", dup_count)

# Optional: check duplicates by business key (مثال: نفس المنتج + التاريخ + المندوب + المبلغ)
key_cols = ["Product_ID", "Sale_Date", "Sales_Rep", "Region", "Sales_Amount"]
existing_key_cols = [c for c in key_cols if c in df.columns]
if existing_key_cols:
    dup_key = df.duplicated(subset=existing_key_cols).sum()
    print("Duplicate rows by key:", dup_key)





--- Missing values & Duplicates report ---


'No missing values'


Duplicate rows: 0
Duplicate rows by key: 0


# **Date** **parsing** / **standardization**

In [17]:


if "Sale_Date" in df.columns:
    df["Sale_Date"] = pd.to_datetime(df["Sale_Date"], errors="coerce")
    bad_dates = df["Sale_Date"].isna().sum()
    print("\nBad/unparseable dates:", bad_dates)

    df["Sale_Month"] = df["Sale_Date"].dt.to_period("M").astype(str)
    df["Sale_Quarter"] = df["Sale_Date"].dt.to_period("Q").astype(str)







Bad/unparseable dates: 0


# **Derived** **metrics** (**Profit** / **Margin** )

In [18]:

# Profit = (Unit_Price - Unit_Cost) * Quantity_Sold

if set(["Unit_Price", "Unit_Cost", "Quantity_Sold"]).issubset(df.columns):
    df["Profit"] = (df["Unit_Price"] - df["Unit_Cost"]) * df["Quantity_Sold"]

if "Sales_Amount" in df.columns and "Profit" in df.columns:
    df["Profit_Margin"] = np.where(df["Sales_Amount"] != 0, df["Profit"] / df["Sales_Amount"], np.nan)

#  **Save** **New** **version**

In [20]:

clean_path_excel = "sales_dataset_New.xlsx"
clean_path_csv = "sales_dataset_New.csv"

df.to_excel(clean_path_excel, index=False)
df.to_csv(clean_path_csv, index=False)

print("\n Saved:", clean_path_excel, "and", clean_path_csv)
print("Final shape:", df.shape)


 Saved: sales_dataset_New.xlsx and sales_dataset_New.csv
Final shape: (1000, 18)
